In [156]:
import torch

## Step 1 - Verify PyTorch and GPU

Before building the model, verify that the Python environment can import PyTorch and that CUDA is available from the same notebook kernel.

The important check is `torch.cuda.is_available()`. If it prints `True`, tensors and models can be moved to `cuda`. The GPU name check confirms which GPU PyTorch is using.

The model code later uses this pattern:

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
```

That keeps the notebook runnable on CPU, while using the GPU when it is available.


In [157]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')

2.12.1
False
no cuda


## Step 2 - Define the Tiny Vocabulary

This notebook uses a character-level vocabulary with exactly three tokens:

```text
a, b, c
```

`stoi` means string-to-integer. It maps characters to token IDs:

```text
a -> 0
b -> 1
c -> 2
```

`itos` means integer-to-string. It maps token IDs back to characters:

```text
0 -> a
1 -> b
2 -> c
```

`encode()` turns text into token IDs. `decode()` turns token IDs back into text. The basic correctness check is:

```python
decode(encode("abacaba")) == "abacaba"
```

Here `vocab_size = 3` because the model has three possible next-token classes.


In [158]:
stoi_dict = {"a": 0, "b": 1, "c": 2}
itos_dict = {0: "a", 1: "b", 2: "c"}

def stoi(s: str) -> int:
    return stoi_dict[s]

def itos(i: int) -> str:
    return itos_dict[i]

def encode(word: str) -> list[int]:
    return [stoi(s) for s in word]

def decode(nums: list[int]) -> str:
    return ''.join(itos(i) for i in nums)

vocab = ['a', 'b', 'c']
vocab_size = len(vocab)

In [159]:
enc = encode("abca")
dec = decode(enc)

print(enc, dec)

[0, 1, 2, 0] abca


## Step 3 - Create a Tiny Training Corpus

The training corpus is a fixed string containing only `a`, `b`, and `c`:

```python
text = "abacabaacbbccabacabaacbbcc"
```

The model does not train directly on characters. It trains on integer token IDs, so the text is encoded and stored as a one-dimensional PyTorch tensor:

```python
data = torch.tensor(encode(text), dtype=torch.long)
```

`torch.long` is used because token IDs are integer class/index values. Embedding layers and cross-entropy targets expect integer token IDs, not floating point vectors.

Useful checks:

```text
data.ndim == 1
data.min() == 0
data.max() == 2
decode(data.tolist()) == text
```


In [160]:
text = "abacabaacbbccabacabaacbbcc"
data = torch.tensor(encode(text), dtype=torch.long)
data

tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])

In [161]:
print(text)
print(encode(text))
print(data)
print(data.shape)
print(data.dtype)
print(data.min().item(), data.max().item())
print(decode(data.tolist()))

abacabaacbbccabacabaacbbcc
[0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2]
tensor([0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 2, 2, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1,
        2, 2])
torch.Size([26])
torch.int64
0 2
abacabaacbbccabacabaacbbcc


In [162]:
decode(data.tolist()) == text

True

## Step 4 - Build Context-Target Examples

For the first baseline, each training example is:

```text
previous block_size tokens -> one next token
```

With `block_size = 4`:

```text
context: a b a c
target:          a
```

Numerically:

```text
x = [0, 1, 0, 2]
y = 0
```

This `block_size` is the toy version of a GPT context window. It controls how many previous token positions the model can see. It is different from `vocab_size`:

```text
vocab_size = how many token types exist = 3
block_size = how many token positions are visible = 4 here
```

`get_batch()` samples random start positions from `data`, then stacks the resulting examples:

```text
xb.shape == [batch_size, block_size]
yb.shape == [batch_size]
```

For this first baseline, `yb` has one target token per context.


In [163]:
block_size = 4
batch_size = 8

def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + block_size] for i in ix])
    return xb, yb

xb, yb = get_batch()

print(xb.shape)  # should be [batch_size, block_size]
print(yb.shape)  # should be [batch_size]

for i in range(batch_size):
    context = decode(xb[i].tolist())
    next_char = decode([yb[i].item()])
    print(context, "->", next_char)

torch.Size([8, 4])
torch.Size([8])
caba -> c
aacb -> b
caba -> c
abaa -> c
acab -> a
baac -> b
ccab -> a
baac -> b


## Step 5 - Train a Simple Baseline Model

Before building GPT-style attention, this baseline learns:

```text
given 4 previous tokens -> predict the next token
```

The model path is:

```text
token IDs -> token embeddings -> flattened context -> linear layer -> logits
```

`n_embd = 8` is an arbitrary small embedding size. It means each token ID becomes 8 learned numbers. With `vocab_size = 3`, the embedding table has:

```text
3 * 8 = 24 learned parameters
```

For a batch shaped `[8, 4]`, the embedding layer produces:

```text
[batch_size, block_size] -> [8, 4, 8]
```

Flattening turns each context into one vector:

```text
[8, 4, 8] -> [8, 32]
```

The final linear layer maps each 32-number context vector to 3 raw scores:

```text
[8, 32] -> [8, 3]
```

Those raw scores are logits for `a`, `b`, and `c`. `F.cross_entropy(logits, targets)` compares the logits with the correct next-token IDs. Do not apply softmax before `cross_entropy`; the loss function expects raw logits.

The training loop repeats this sequence:

```text
sample batch -> forward pass -> compute loss -> clear gradients -> backward pass -> optimizer step
```


In [164]:
import torch.nn as nn
import torch.nn.functional as F

n_embd = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

class ContextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(block_size * n_embd, vocab_size)

    def forward(self, idx, targets=None):
        x = self.token_embedding(idx)   # [B, T, C]
        B, T, C = x.shape               # C = n_embd
        x = x.reshape(B, T * C)         # [B, T*C]
        logits = self.lm_head(x)        # [B, V]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [165]:
model = ContextModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(1000+1):
    xb, yb = get_batch()
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 1.2997052669525146
100 0.22312197089195251
200 0.03417773172259331
300 0.024391397833824158
400 0.09016011655330658
500 0.07551261782646179
600 0.008867588825523853
700 0.184119313955307
800 0.08747980743646622
900 0.15081395208835602
1000 0.07274399697780609


## Step 6 - Generate Text from the Baseline

Generation repeatedly predicts one token and appends it to the sequence.

At each step, the model receives only the last `block_size` tokens:

```python
idx_cond = idx[-block_size:]
```

Then the shape is changed from one sequence to one batch of one sequence:

```text
[T] -> [1, T]
```

The model returns logits for the next token. Softmax converts those logits into probabilities over the three possible tokens:

```text
probability of a
probability of b
probability of c
```

`torch.multinomial()` samples from those probabilities. The sampled token is appended to the running sequence, and the process repeats.

`@torch.no_grad()` disables gradient tracking during generation, because generation is inference, not training. `model.eval()` switches the model into evaluation mode, and `model.train()` switches it back afterward.


In [166]:
@torch.no_grad()
def generate(model, start, max_new_tokens):
    model.eval()

    idx = torch.tensor(encode(start), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[-block_size:].unsqueeze(0)   # last block_size tokens -> [1, T]

        logits, _ = model(idx_cond)                 # [1, V]
        probs = F.softmax(logits, dim=-1)           # [1, V]

        idx_next = torch.multinomial(probs, num_samples=1)   # [1, 1]
        idx = torch.cat((idx, idx_next.squeeze(0)), dim=0)

    model.train()
    return decode(idx.tolist())

print(generate(model, "abac", 20))

abacabaacbbccabacabaacbb


## Step 7a - Move Toward GPT-Style Training

The baseline predicted only one token after the whole context. GPT-style training predicts the next token at every position inside the block.

The batch target changes from:

```text
yb.shape == [B]
```

to:

```text
yb.shape == [B, T]
```

Example:

```text
xb: a b a c
yb: b a c a
```

So the model learns all of these predictions from one row:

```text
a       -> b
a b     -> a
a b a   -> c
a b a c -> a
```

Shape letters used throughout this notebook:

```text
B = batch size            how many sequences in the batch
T = time / positions      how many token positions, up to block_size
C = channels = n_embd     the hidden width of a token vector
V = vocab_size            how many token types exist
```

`C` always means the embedding width. Logits are the one tensor whose last dimension is `V` rather than `C`, because `lm_head` maps `n_embd -> vocab_size`. Watch for this when unpacking a shape: `B, T, C = x.shape` on an embedding and `B, T, V = logits.shape` on the output are different widths.

The MiniGPT skeleton adds two embeddings:

```text
token embedding:    what token is this?
position embedding: where is this token inside the context window?
```

For input `a b c a`, the two `a` tokens have the same token embedding, but different positional embeddings:

```text
first a  = token_embedding[a] + position_embedding[0]
second a = token_embedding[a] + position_embedding[3]
```

In tensor form:

```python
x = tok_emb + pos_emb
```

with shapes:

```text
tok_emb: [B, T, C]
pos_emb:    [T, C]
result:  [B, T, C]
```

PyTorch broadcasts `pos_emb` across the batch dimension. Addition keeps the hidden size equal to `n_embd`; concatenation would double it and require changing later layers.

The logits now have shape:

```text
[B, T, V]
```

For cross-entropy, the batch and time dimensions are flattened:

```text
logits:  [B, T, V] -> [B*T, V]
targets: [B, T]             -> [B*T]
```


In [167]:
def get_batch():
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    # xb = block_size tokens starting at i          -> [batch_size, block_size]
    xb = torch.stack([data[i:i + block_size] for i in ix])
    # yb = the same window shifted one step forward -> [batch_size, block_size]
    yb = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return xb, yb

get_batch()


(tensor([[0, 1, 0, 0],
         [2, 0, 1, 0],
         [1, 0, 2, 0],
         [2, 1, 1, 2],
         [1, 0, 2, 0],
         [0, 1, 0, 2],
         [2, 1, 1, 2],
         [0, 1, 0, 0]]),
 tensor([[1, 0, 0, 2],
         [0, 1, 0, 0],
         [0, 2, 0, 1],
         [1, 1, 2, 2],
         [0, 2, 0, 1],
         [1, 0, 2, 0],
         [1, 1, 2, 2],
         [1, 0, 0, 2]]))

In [168]:
batch_size = 8
block_size = 8
n_embd = 16
n_head = 2
n_layer = 1

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        # one embedding per position in the context window
        self.position_embedding = nn.Embedding(block_size, n_embd)
        # applied at every position: n_embd -> vocab_size
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        logits = self.lm_head(x)                         # [B, T, V]

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            # logits:  [B, T, V] -> [B*T, V]
            # targets: [B, T]    -> [B*T]
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))

        return logits, loss

model = MiniGPT().to(device)

xb, yb = get_batch()
xb = xb.to(device)
yb = yb.to(device)

logits, loss = model(xb, yb)

print(xb.shape)      # [8, 8]
print(yb.shape)      # [8, 8]
print(logits.shape)  # [8, 8, 3]
print(loss.item())

torch.Size([8, 8])
torch.Size([8, 8])
torch.Size([8, 8, 3])
1.387282371520996


In [169]:
model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(2000):
    xb, yb = get_batch()
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 1.183016061782837
100 1.1059746742248535
200 1.0630155801773071
300 1.0512160062789917
400 0.9748817682266235
500 1.0998945236206055
600 1.0210105180740356
700 1.0525609254837036
800 1.0568363666534424
900 1.054112434387207
1000 0.9508479833602905
1100 1.0181549787521362
1200 1.06044340133667
1300 1.0364238023757935
1400 1.0327143669128418
1500 1.0458310842514038
1600 1.0254065990447998
1700 1.042965292930603
1800 0.988623321056366
1900 1.0198545455932617


In [170]:
@torch.no_grad()
def generate(model, start, max_new_tokens):
    model.eval()

    idx = torch.tensor(encode(start), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[-block_size:].unsqueeze(0)   # last block_size tokens -> [1, T]

        logits, _ = model(idx_cond)                 # [1, T, V]
        logits = logits[:, -1, :]                   # keep the last position -> [1, V]

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)   # [1, 1]
        idx = torch.cat((idx, idx_next.squeeze(0)), dim=0)

    model.train()
    return decode(idx.tolist())

print(generate(model, "abac", 20))

abacbaabbcabaccababcbaca


## Step 7b - Add One Causal Self-Attention Head

Causal self-attention lets each token position mix information from previous positions, while preventing it from looking at future positions.

For a sequence:

```text
a b a c
```

allowed visibility is:

```text
position 0 sees: a
position 1 sees: a b
position 2 sees: a b a
position 3 sees: a b a c
```

The attention head learns three projections from the same input `x`:

```text
query: what is this position looking for?
key:   what does each position contain?
value: what information should be copied forward?
```

The query-key product computes attention scores between all pairs of positions:

```python
wei = q @ k.transpose(-2, -1)
```

Shape:

```text
[B, T, head_size] @ [B, head_size, T] -> [B, T, T]
```

The scale factor:

```python
wei = wei * (head_size ** -0.5)
```

keeps the dot-product magnitudes more stable as `head_size` changes.

The lower-triangular mask is:

```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

Zeros are replaced with `-inf` before softmax:

```python
wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
```

After softmax, future positions receive probability zero. The final output is a weighted mixture of value vectors:

```python
out = wei @ v
```

Shape:

```text
[B, T, T] @ [B, T, head_size] -> [B, T, head_size]
```

`register_buffer()` stores the causal mask as part of the module state without making it a trainable parameter. When the model moves to GPU, the buffer moves with it.


In [171]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # causal mask, stored as a buffer so it moves with the model but is not trained
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape    # C = n_embd

        k = self.key(x)      # [B, T, head_size]
        q = self.query(x)    # [B, T, head_size]

        # [B, T, hs] @ [B, hs, T] -> [B, T, T]
        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.shape[-1] ** -0.5)   # keeps the softmax out of saturation

        # -inf, not 0: the softmax then renormalizes over the visible prefix only.
        # tril is cropped to [:T, :T] because T < block_size while generating.
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )
        wei = F.softmax(wei, dim=-1)    # [B, T, T]

        v = self.value(x)               # [B, T, head_size]
        out = wei @ v                   # [B, T, head_size]

        return out

In [172]:
batch_size = 8
block_size = 8
n_embd = 16
n_head = 2
n_layer = 1

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.sa = Head(n_embd)   # one head, head_size = n_embd

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        x = self.sa(x)                                   # [B, T, C]
        logits = self.lm_head(x)                         # [B, T, V]

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, V),
                targets.view(B * T),
            )

        return logits, loss

model = MiniGPT().to(device)

xb, yb = get_batch()
xb = xb.to(device)
yb = yb.to(device)

print(xb.shape, yb.shape)

logits, loss = model(xb, yb)

print(logits.shape)  # [batch_size, block_size, vocab_size]
print(loss.item())
print(model.sa.tril[:4, :4])

torch.Size([8, 8]) torch.Size([8, 8])
torch.Size([8, 8, 3])
1.1417081356048584
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [173]:
model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(2000):
    xb, yb = get_batch()
    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 1.1266947984695435
100 0.5567439794540405
200 0.4411633610725403
300 0.7105921506881714
400 0.7537102699279785
500 0.5356435775756836
600 0.5812129378318787
700 0.49266475439071655
800 0.4627659022808075
900 0.6584315299987793
1000 0.5137654542922974
1100 0.5155761241912842
1200 0.4621179401874542
1300 0.5682400465011597
1400 0.5638474225997925
1500 0.4957835376262665
1600 0.4927155375480652
1700 0.7685296535491943
1800 0.5795254707336426
1900 0.463131844997406


In [174]:
# generate() is unchanged from Step 7a
print("sample: ", generate(model, "abac", 22))
print("target: ", text)

sample:  abacabacabacabacaabcabbcab
target:  abacabaacbbccabacabaacbbcc


## Step 8 - Multi-Head Attention

Step 7b used one attention head with `head_size = n_embd`. A single head learns one way of relating positions. **Multi-head attention** runs several smaller heads in parallel, each free to focus on a different pattern (for this corpus, e.g. "what was the previous token?" vs. "where am I in the repeating block?").

The embedding width is *split* across the heads instead of being duplicated:

```text
head_size = n_embd // n_head = 16 // 2 = 8
```

Each head (reusing the `Head` class from Step 7b) produces `[B, T, head_size]`. Concatenating all heads along the last dimension rebuilds the full width:

```text
2 heads x 8  ->  [B, T, 16]
```

A final linear layer (`proj`) mixes the concatenated heads back together. Keeping the output width equal to `n_embd` is what lets attention be *added* to `x` in the residual connection later (Step 10) without changing any other layer.

`nn.ModuleList` is used instead of a plain Python list so PyTorch registers every head's parameters - they appear in `model.parameters()` and move to the GPU with the model. A plain list would hide them from the optimizer.

In [175]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        # nn.ModuleList, not a plain list, so the heads' parameters are registered
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # [B, T, n_head*head_size]
        out = self.proj(out)                                 # [B, T, n_embd]
        return out

## Step 9 - Feed-Forward Network

Attention moves information *between* positions - its output is a weighted average of value vectors, which is close to linear. The feed-forward network adds per-position, non-linear computation: it processes each token independently and transforms what attention just gathered.

The standard shape is expand -> non-linearity -> project back:

```text
[B, T, 16] -> [B, T, 64] -> ReLU -> [B, T, 16]
```

The inner width is `4 * n_embd` by convention (the same 4x ratio used in GPT-2). The `ReLU` is the block's only non-linearity; without it, two stacked linear layers would collapse into a single linear map and the network could not learn non-linear structure.

In [176]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # expand -> ReLU -> project back; ReLU is the block's only non-linearity
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

## Step 10 - Transformer Block: Residuals + Layer Norm

A transformer block combines the two previous pieces - attention (Step 8) and feed-forward (Step 9) - with two techniques that make deep networks trainable.

**Residual connections.** Instead of replacing `x`, each sub-layer *adds* to it:

```python
x = x + self.sa(...)
x = x + self.ffwd(...)
```

The `x +` path is a gradient "highway": during backpropagation gradients flow straight back through the addition, so even deep stacks keep a strong learning signal. Each sub-layer only has to learn a *correction* to `x`, not rebuild it from scratch.

**Layer normalization.** `nn.LayerNorm(n_embd)` normalizes each token vector to mean 0 / variance 1 across its 16 features, then rescales with learned parameters. This keeps activations at a stable scale so training does not blow up or stall.

This block uses the **pre-norm** arrangement - normalize *before* each sub-layer - which is what GPT-2 and later models use:

```python
x = x + self.sa(self.ln1(x))     # communicate between positions
x = x + self.ffwd(self.ln2(x))   # think at each position
```

"Communicate, then think" is a compact summary of the whole transformer block.

In [177]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head   # the width is split across heads, not duplicated
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # pre-norm residual around attention   (communicate between positions)
        x = x + self.sa(self.ln1(x))
        # pre-norm residual around feed-forward (think at each position)
        x = x + self.ffwd(self.ln2(x))
        return x

## Step 11 - Assemble the Full MiniGPT

The full model stacks everything built so far:

```text
token IDs
  -> token embedding + position embedding
  -> n_layer transformer blocks (stacked)
  -> final layer norm
  -> linear head  ->  logits [B, T, V]
```

`nn.Sequential(*[Block(...) for _ in range(n_layer)])` stacks `n_layer` identical blocks. Here `n_layer = 2`, so the network is genuinely deep - this is where the residual connections and layer norm from Step 10 start to matter.

`ln_f` is a final layer norm applied after the blocks and before the output head (standard in GPT-2). The head then maps each position's `n_embd = 16`-number vector to `vocab_size = 3` logits, one per token type.

Nothing about the loss changes from Step 7a: logits are `[B, T, V]`, and cross-entropy flattens the batch and time dimensions into `[B*T, V]` vs `[B*T]`.

In [213]:
block_size = 4
batch_size = 8
n_embd = 16
n_head = 4
n_layer = 4   # stacked blocks: the model is now deep

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)   # final norm, before the output head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)              # [B, T, C]
        pos = torch.arange(T, device=idx.device)         # [T]
        pos_emb = self.position_embedding(pos)           # [T, C]

        x = tok_emb + pos_emb                            # [B, T, C]
        x = self.blocks(x)                               # [B, T, C]
        x = self.ln_f(x)                                 # [B, T, C]
        logits = self.lm_head(x)                         # [B, T, V]

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))

        return logits, loss


model = MiniGPT().to(device)

xb, yb = get_batch()
xb, yb = xb.to(device), yb.to(device)
logits, loss = model(xb, yb)

n_params = sum(p.numel() for p in model.parameters())
print("logits:", tuple(logits.shape))   # [batch_size, block_size, vocab_size]
print("loss:", loss.item())
print("parameters:", n_params)

logits: (8, 4, 3)
loss: 1.1467841863632202
parameters: 13123


## Step 12 - Train the Full Model and Generate

Nothing about the training loop changes from Step 7a - sample a batch, forward, compute loss, zero gradients, backward, optimizer step. Only the model is bigger.

Because the corpus is a short repeating pattern (`abacabaacbbcc` twice), a correctly wired model should drive the loss well below `1.0` and then reproduce that pattern almost deterministically when generating.

If the loss does *not* fall on this tiny dataset, the bug is almost always in one of: the batch targets, the causal mask, the loss reshape, or the optimizer step. This is the intentional-overfitting check described in the "Later Steps" note: a model that cannot memorize a trivial corpus has a pipeline bug, not a capacity problem.

In [214]:
model = MiniGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(3000):
    xb, yb = get_batch()
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

print("final loss:", loss.item())

0 1.2396514415740967
100 0.44680771231651306
200 0.4135325253009796
300 0.3843989074230194
400 0.391666054725647
500 0.2892860770225525
600 0.483783096075058
700 0.41252002120018005
800 0.6690346002578735
900 0.424145370721817
1000 0.392300009727478
1100 0.3794638514518738
1200 0.3708595931529999
1300 0.4092768430709839
1400 0.37250834703445435
1500 0.4403360188007355
1600 0.3534293472766876
1700 0.3685409128665924
1800 0.3905545175075531
1900 0.397478848695755
2000 0.4094037711620331
2100 0.3797023892402649
2200 0.37480399012565613
2300 0.32805007696151733
2400 0.30027279257774353
2500 0.4545184373855591
2600 0.3143397569656372
2700 0.37914538383483887
2800 0.39428025484085083
2900 0.40655356645584106
final loss: 0.3399660289287567


In [215]:
@torch.no_grad()
def generate(model, start, max_new_tokens):
    model.eval()

    idx = torch.tensor(encode(start), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[-block_size:].unsqueeze(0)   # last block_size tokens -> [1, T]

        logits, _ = model(idx_cond)                 # [1, T, V]
        logits = logits[:, -1, :]                   # keep the last position -> [1, V]

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)   # [1, 1]

        for p, v in zip(probs.squeeze(0).tolist(), vocab):
            n = decode(idx_next.squeeze(0).tolist())
            if v == n:
                print("{:.3f}".format(p), v, " <-")
            else:
                print("{:.3f}".format(p), v)    
        print()
        idx = torch.cat((idx, idx_next.squeeze(0)), dim=0)

    model.train()
    return decode(idx.tolist())

print("sample: ", generate(model, "abac", 22))
print("target: ", text)

0.998 a  <-
0.000 b
0.002 c

0.000 a
0.999 b  <-
0.000 c

0.998 a  <-
0.001 b
0.001 c

0.667 a  <-
0.002 b
0.331 c

0.001 a
0.000 b
0.999 c  <-

0.001 a
0.999 b  <-
0.000 c

0.001 a
0.999 b  <-
0.000 c

0.001 a
0.000 b
0.999 c  <-

0.001 a
0.000 b
0.998 c  <-

0.997 a  <-
0.000 b
0.003 c

0.002 a
0.998 b  <-
0.000 c

0.998 a  <-
0.001 b
0.001 c

0.667 a  <-
0.002 b
0.331 c

0.001 a
0.000 b
0.999 c  <-

0.001 a
0.999 b  <-
0.000 c

0.001 a
0.999 b  <-
0.000 c

0.001 a
0.000 b
0.999 c  <-

0.001 a
0.000 b
0.998 c  <-

0.997 a  <-
0.000 b
0.003 c

0.002 a
0.998 b  <-
0.000 c

0.998 a  <-
0.001 b
0.001 c

0.667 a  <-
0.002 b
0.331 c

sample:  abacabaacbbccabaacbbccabaa
target:  abacabaacbbccabacabaacbbcc


## Later Steps - Compare, Overfit, Then Vary One Thing

After single-head causal attention works, useful next checks are:

```text
frequency baseline
embedding + linear baseline
MiniGPT with causal attention
```

On this tiny dataset, the goal is not benchmark performance. The goal is to verify that each model can learn the simple next-token structure and to understand what each added component changes.

A good debugging target is intentional overfitting: use the tiny fixed corpus and train until the loss becomes very low. If the model cannot memorize a tiny dataset, something is probably wrong in the data pipeline, masking, loss shape, or optimization loop.

After that, vary one thing at a time:

```text
block_size
corpus length
embedding size
sampling temperature
number of heads/layers
```

Changing one variable at a time makes failures easier to attribute.


## Current Implementation Status

The full character-level MiniGPT is now complete. Implemented:

```text
PyTorch/GPU verification
3-token character vocabulary
encoded training data
baseline context-target batching
embedding + linear baseline
sampling-based generation
GPT-style sequence targets
token + position embeddings
single causal self-attention head
multi-head attention
feed-forward block
residual connections
layer normalization
stacked transformer blocks
final layer norm + output head
corrected sequence-aware generation
full GPT training loop
```

This is every component of a real GPT, in miniature. Going from this toy to a practical model means *scaling the same pieces* - a larger `vocab_size` (a real sub-word tokenizer), a longer `block_size`, a bigger `n_embd`, more heads and layers, and a much larger corpus - plus training conveniences (dropout, a learning-rate schedule, weight-decay tuning). The architecture itself does not change.